# Hybrid Information Retrieval System for EnronQA
## BM25S + Dense (BGE) + RRF Fusion + Jina Reranker v3 (LoRA Fine-tuned)

**Dataset**: [MichaelR207/enron_qa_0922](https://huggingface.co/datasets/MichaelR207/enron_qa_0922) — 73,772 raw Enron emails  
**Eval**: 500 queries from held-out 20% split, single-document qrels  
**Metrics**: MRR, NDCG@{1,5,10,20,50,100}, Precision@k, Recall@k  
**Target**: NDCG@10 ≥ 0.75

### Pipeline Components
1. **BM25S** (bm25s) over tokenized email text with subject boosting
2. **Dense retrieval**: BAAI/bge-base-en-v1.5 (768-dim), FAISS HNSW index, BGE query-instruction prefix
3. **Hybrid fusion**: Weighted Reciprocal Rank Fusion (RRF)
4. **Jina Reranker v3**: `jinaai/jina-reranker-v3` (0.6B listwise LBNL causal self-attention model)
5. **LoRA Fine-tuning**: Pointwise sequence classification tuning on EnronQA domain data using PEFT adapters

### Key References
- Wang et al. (2025). *jina-reranker-v3: Last but Not Late Interaction for Listwise Document Reranking.* arXiv:2509.25085v4
- Nogueira & Cho (2019). *Passage Re-ranking with BERT.* arXiv:1901.04085
- Xiao et al. (2023). *C-Pack: Packaged Resources for General Chinese Embeddings.* arXiv:2309.07597 (BGE models)
- Cormack et al. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods.* SIGIR 2009
- Robertson & Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in IR
- Karpukhin et al. (2020). *Dense Passage Retrieval for Open-Domain QA.* EMNLP 2020 (hard negative mining)
- Xiong et al. (2021). *Approximate Nearest Neighbor Negative Contrastive Learning.* ICLR 2021 (negative sampling balance)
- Thakur et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of IR Models.* NeurIPS 2021

---
## Section 1 — Environment Setup

In [ ]:
# ── Install dependencies (run once) ────────────────────────────────────────────
!pip install -q datasets bm25s sentence-transformers faiss-gpu nltk \
    matplotlib seaborn tqdm numpy pandas scikit-learn torch transformers peft accelerate

---
## Section 2 — Imports & Configuration

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import re
import math
import time
import pickle
import hashlib
import logging
import warnings
import random
from typing import Dict, List, Tuple, Optional, Set
from collections import defaultdict, Counter

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
torch.set_num_threads(os.cpu_count())
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch.nn.functional as F

# ── HuggingFace / PEFT / Transformers ──────────────────────────────────────────
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# ── BM25S ─────────────────────────────────────────────────────────────────────
import bm25s

# ── FAISS ─────────────────────────────────────────────────────────────────────
import faiss
faiss.omp_set_num_threads(os.cpu_count())

# ── NLTK ──────────────────────────────────────────────────────────────────────
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ── Logging / Reproducibility ─────────────────────────────────────────────────
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)s  %(message)s')
logger = logging.getLogger(__name__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Global Hyperparameters ────────────────────────────────────────────────────
DATA_DIR       = "./data"
CACHE_DIR      = "./cache"
TOP_K          = 100
EVAL_K_VALUES  = [1, 5, 10, 20, 50, 100]

# Dense retrieval
SBERT_MODEL    = "BAAI/bge-base-en-v1.5"
BGE_QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
BATCH_SIZE     = 512
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# BM25 defaults (tuned later via grid search)
BM25_K1        = 1.5
BM25_B         = 0.75

# RRF / fusion defaults (tuned later)
RRF_K          = 60
BM25_WEIGHT    = 0.5
DENSE_WEIGHT   = 0.5
FETCH_K        = 200

# Chunking
DENSE_CHUNK_SIZE    = 200
DENSE_CHUNK_OVERLAP = 50

# Jina Reranker
JINA_RERANKER_MODEL  = "jinaai/jina-reranker-v3"
RERANKER_TOP_N       = 50
JINA_MAX_LENGTH      = 1024
JINA_BATCH_DOCS      = 32

# LoRA Fine-tuning
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGET      = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
FT_LEARNING_RATE = 2e-4
FT_NUM_EPOCHS    = 3
FT_BATCH_SIZE    = 8
FT_WARMUP_RATIO  = 0.1
FT_MAX_LENGTH    = 512
FT_GRAD_ACCUM    = 4
FT_OUTPUT_PATH   = os.path.join(CACHE_DIR, "jina-reranker-v3-enronqa-lora")

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU VRAM: {vram_gb:.1f} GB")
    if vram_gb < 8:
        print("WARNING: <8GB VRAM detected. Reducing FT_BATCH_SIZE to 4 and FT_GRAD_ACCUM to 8.")
        FT_BATCH_SIZE = 4
        FT_GRAD_ACCUM = 8

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Device          : {DEVICE}")
print(f"SBERT model     : {SBERT_MODEL}")
print(f"Reranker model  : {JINA_RERANKER_MODEL}")
print(f"Top-K retrieval : {TOP_K}")
print(f"Fetch-K (pool)  : {FETCH_K}")

---
## Section 3 — Dataset Loading (EnronQA)

In [ ]:
def load_enronqa(split_ratio: float = 0.2, seed: int = 42, max_queries: int = 500):
    """
    Load EnronQA dataset and create corpus, queries, and qrels.

    Schema:
      email     : full raw email string (includes Subject:, Sender:, body)
      questions : list of question strings
      path      : file path — used as the unique email id
      user      : mailbox owner

    Returns:
      corpus_raw, queries_raw, qrels, train_pairs (for cross-encoder fine-tuning)
    """
    ds = load_dataset("MichaelR207/enron_qa_0922", split="train")

    # ── Build corpus ──────────────────────────────────────────────────
    corpus_raw = {}
    for row in ds:
        eid = row["path"]
        email_text = row["email"] or ""
        subject = ""
        for line in email_text.splitlines():
            if line.startswith("Subject:"):
                subject = line.replace("Subject:", "").strip()
                break
        corpus_raw[eid] = {"title": subject, "text": email_text}

    # ── Hash-based deterministic train/test split ─────────────────────
    all_eids = list(corpus_raw.keys())
    test_eids = set()
    for eid in all_eids:
        bucket = int(hashlib.md5(eid.encode()).hexdigest(), 16) % 100
        if bucket < int(split_ratio * 100):
            test_eids.add(eid)

    # ── Build queries + qrels from TEST emails only ───────────────────
    queries_raw = {}
    qrels = {}
    q_counter = 0

    for row in ds:
        if q_counter >= max_queries:
            break
        eid = row["path"]
        if eid not in test_eids:
            continue
        for q_text in (row.get("questions") or []):
            if q_counter >= max_queries:
                break
            if not q_text or len(q_text.split()) < 3:
                continue
            qid = f"q_{q_counter:06d}"
            queries_raw[qid] = q_text
            qrels[qid] = {eid: 1}
            q_counter += 1

    # ── Build training pairs from TRAIN emails (for cross-encoder fine-tuning) ───
    train_pairs = []
    for row in ds:
        eid = row["path"]
        if eid in test_eids:
            continue  # Skip test split
        for q_text in (row.get("questions") or []):
            if not q_text or len(q_text.split()) < 3:
                continue
            train_pairs.append({"query": q_text, "doc_id": eid})

    return corpus_raw, queries_raw, qrels, train_pairs

corpus_raw, queries_raw, qrels, train_pairs_raw = load_enronqa()

print(f"EnronQA loaded successfully")
print(f"  Corpus size       : {len(corpus_raw):,} emails")
print(f"  Test queries      : {len(queries_raw):,}")
print(f"  Qrel entries      : {sum(len(v) for v in qrels.values()):,} query-doc pairs")
print(f"  Training pairs    : {len(train_pairs_raw):,} (for fine-tuning)")

---
## Section 4 — Dataset Statistics

In [ ]:
doc_lengths = [
    len((v.get('title', '') + ' ' + v.get('text', '')).split())
    for v in corpus_raw.values()
]

stats = {
    "Number of documents"        : len(corpus_raw),
    "Number of queries"          : len(queries_raw),
    "Avg document length (words)": round(np.mean(doc_lengths), 1),
    "Max document length (words)": max(doc_lengths),
    "Min document length (words)": min(doc_lengths),
    "Avg relevant docs per query": round(np.mean([len(v) for v in qrels.values()]), 2),
}
df_stats = pd.DataFrame(list(stats.items()), columns=["Metric", "Value"])
print("\n=== Dataset Statistics ===")
print(df_stats.to_string(index=False))

---
## Section 5 — Email-Specific Preprocessing Pipeline

Key design decisions for the Enron email domain:
1. **Preserve forwarded/quoted content**: EnronQA questions frequently reference content inside forwarded blocks
2. **Subject boosting for BM25**: Repeat subject 3× in BM25 corpus — subjects are the most discriminative field
3. **Sender/recipient boosting**: Repeat From/To fields 2× — many queries ask about specific people
4. **Thread-aware chunking**: Split at message boundaries (forwarded/original message markers) before word-level chunking
5. **Email address tokenization**: Keep email addresses as matchable tokens alongside decomposed parts

In [ ]:
class EnronEmailProcessor:
    """
    Email-specific preprocessing pipeline for the EnronQA IR system.

    Improvements over generic scientific-text preprocessing:
    - Parses email structure (From/To/Subject/body)
    - Preserves forwarded/quoted content (answer-bearing for EnronQA)
    - Thread-aware chunking at message boundaries
    - Email address handling (keeps full address + decomposed parts)
    - Subject/sender/recipient field boosting for BM25
    """

    _PREAMBLE_SPLIT = re.compile(r'={5,}\s*\n')
    _FWD_PATTERN = re.compile(
        r'-{3,}\s*(?:Forwarded|Original Message)\s*-{3,}',
        re.IGNORECASE
    )
    _HEADER_PATTERN = re.compile(
        r'^(From|To|Cc|Sent|Date|Subject):\s*(.+)',
        re.IGNORECASE | re.MULTILINE
    )
    _EMAIL_PATTERN = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')

    def __init__(self, remove_stopwords: bool = True, min_token_len: int = 2):
        self.remove_stopwords = remove_stopwords
        self.min_token_len = min_token_len
        self._stopwords = set(stopwords.words('english')) if remove_stopwords else set()

    # ── Core text cleaning ────────────────────────────────────────────
    def preprocess_document(self, text: str) -> str:
        if not text:
            return ""
        # Extract email addresses before cleaning
        emails = self._EMAIL_PATTERN.findall(text)
        email_parts = []
        for email in emails:
            local, domain = email.split('@', 1)
            # Keep both the full email and decomposed name parts
            email_parts.extend([email.replace('@', ' at '), local.replace('.', ' ')])

        # Replace non-alphanumeric chars except hyphens with spaces
        text = re.sub(r'[^\w\s\-]', ' ', text)
        tokens = text.split()
        filtered = []
        for tok in tokens:
            tok_lower = tok.lower()
            if re.fullmatch(r'\d+', tok_lower):
                continue
            if self.remove_stopwords and tok_lower in self._stopwords:
                continue
            if len(tok_lower) < self.min_token_len:
                continue
            filtered.append(tok_lower)

        # Add decomposed email parts
        for part in email_parts:
            for tok in part.lower().split():
                if len(tok) >= self.min_token_len and tok not in self._stopwords:
                    filtered.append(tok)

        return ' '.join(filtered)

    def tokenize(self, text: str) -> List[str]:
        preprocessed = self.preprocess_document(text)
        return preprocessed.split() if preprocessed else []

    # ── Email structure parsing ───────────────────────────────────────
    def clean_email_body(self, email_text: str) -> str:
        """Strip only the leading EnronQA metadata preamble. Keep all forwarded/quoted content."""
        if not email_text:
            return ""
        parts = self._PREAMBLE_SPLIT.split(email_text, maxsplit=1)
        if len(parts) == 2:
            body = parts[1].strip()
            if len(body.split()) >= 5:
                return body
        return email_text.strip()

    def parse_email_fields(self, email_text: str) -> dict:
        """Extract structured fields from raw email text."""
        fields = {'from': '', 'to': '', 'cc': '', 'subject': '', 'body': ''}
        body = self.clean_email_body(email_text)
        for match in self._HEADER_PATTERN.finditer(body[:2000]):  # Search headers in first 2K chars
            key = match.group(1).lower()
            if key in ('sent', 'date'):
                continue  # Skip date headers
            if key in fields:
                fields[key] += ' ' + match.group(2).strip()
        fields['body'] = body
        return fields

    # ── BM25 corpus construction ──────────────────────────────────────
    def combine_fields_bm25(self, doc: dict) -> str:
        """
        For BM25: weight subject 3×, sender/recipient 2×, body 1×.
        This boosts the most discriminative fields for lexical matching.
        """
        subject = doc.get('title', '') or ''
        raw = doc.get('text', '') or ''
        fields = self.parse_email_fields(raw)

        parts = []
        parts.extend([subject] * 3)          # Subject 3×
        if fields['from'].strip():
            parts.extend([fields['from']] * 2)   # Sender 2×
        if fields['to'].strip():
            parts.extend([fields['to']] * 2)     # Recipients 2×
        parts.append(fields['body'])              # Body 1×
        return ' '.join(parts).strip()

    # ── Dense corpus construction ─────────────────────────────────────
    def combine_fields_dense(self, doc: dict) -> str:
        """For dense embedding: subject + cleaned body (with forwarded content preserved)."""
        subject = doc.get('title', '') or ''
        raw = doc.get('text', '') or ''
        body = self.clean_email_body(raw)
        if len(body.split()) < 5:
            body = raw
        return f"{subject}\n{body}".strip()

    # ── Thread-aware chunking ─────────────────────────────────────────
    def chunk_text(self, text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
        """
        Thread-aware chunking: split at forwarded/original message boundaries first,
        then apply word-level chunking within each message segment.
        """
        segments = self._FWD_PATTERN.split(text)
        all_chunks = []
        for segment in segments:
            segment = segment.strip()
            if not segment:
                continue
            words = segment.split()
            if len(words) <= chunk_size:
                if words:
                    all_chunks.append(segment)
            else:
                start = 0
                while start < len(words):
                    end = start + chunk_size
                    all_chunks.append(' '.join(words[start:end]))
                    if end >= len(words):
                        break
                    start += chunk_size - overlap
        return all_chunks if all_chunks else ([text] if text.strip() else [])

    # ── Full corpus preprocessing ─────────────────────────────────────
    def preprocess_corpus(self, corpus: Dict[str, dict]):
        """Preprocess corpus for BM25 and build chunked dense corpus."""
        doc_ids, tokenized_corpus = [], []
        chunk_texts, chunk_to_doc = [], []

        for doc_id, doc in tqdm(corpus.items(), desc='Preprocessing corpus'):
            # BM25 corpus
            combined = self.combine_fields_bm25(doc)
            tokens = self.tokenize(combined)
            doc_ids.append(doc_id)
            tokenized_corpus.append(tokens)

            # Dense chunked corpus
            dense_text = self.combine_fields_dense(doc)
            chunks = self.chunk_text(dense_text, chunk_size=DENSE_CHUNK_SIZE, overlap=DENSE_CHUNK_OVERLAP)
            if not chunks:
                chunks = [doc.get('title', '') or '']
            for c in chunks:
                chunk_texts.append(c)
                chunk_to_doc.append(doc_id)

        return doc_ids, tokenized_corpus, chunk_texts, chunk_to_doc


# ── Instantiate and run preprocessing ────────────────────────────────
processor = EnronEmailProcessor(remove_stopwords=True, min_token_len=2)
doc_ids, tokenized_corpus, chunk_texts, chunk_to_doc = processor.preprocess_corpus(corpus_raw)

# Statistics
token_lengths = [len(t) for t in tokenized_corpus]
chunk_lengths = [len(c.split()) for c in chunk_texts]
chunks_per_doc = list(Counter(chunk_to_doc).values())

print(f"\n=== Preprocessing Statistics ===")
print(f"  Documents          : {len(doc_ids):,}")
print(f"  Avg tokens/doc     : {np.mean(token_lengths):.1f}")
print(f"  Total chunks       : {len(chunk_texts):,}")
print(f"  Avg words/chunk    : {np.mean(chunk_lengths):.1f}")
print(f"  Avg chunks/doc     : {np.mean(chunks_per_doc):.2f}")
print(f"  Docs with >1 chunk : {sum(1 for c in chunks_per_doc if c > 1)} ({100*sum(1 for c in chunks_per_doc if c > 1)/len(chunks_per_doc):.1f}%)")

---
## Section 6 — Chunking Sanity Check

In [ ]:
print("=== Chunking Sanity Check ===")
print(f"chunk_to_doc length match : {len(chunk_texts) == len(chunk_to_doc)}")
print(f"Unique docs in chunk_to_doc: {len(set(chunk_to_doc)):,}")
print(f"Chunks with <5 words: {sum(1 for l in chunk_lengths if l < 5)} "
      f"({100*sum(1 for l in chunk_lengths if l < 5)/len(chunk_lengths):.1f}%)")

# Show sample documents
print("\n" + "="*90)
print("SAMPLE DOCUMENTS — raw vs dense vs first chunk")
print("="*90)

for did in random.sample(doc_ids, 3):
    dense_text = processor.combine_fields_dense(corpus_raw[did])
    doc_chunks = [chunk_texts[i] for i in range(len(chunk_texts)) if chunk_to_doc[i] == did]
    print(f"\n--- doc_id: {did} ---")
    print(f"Dense text length : {len(dense_text.split())} words")
    print(f"Num chunks        : {len(doc_chunks)}")
    print(f"First chunk (200 chars): {doc_chunks[0][:200]!r}")

---
## Section 7 — Sparse Retriever (BM25S)

Using the `bm25s` library which achieves massive speedups via sparse matrix vector operations.
BM25 scoring:
$$score(q,d) = \sum_{t \in q} IDF(t) \cdot \frac{tf(t,d) \cdot (k_1 + 1)}{tf(t,d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{avgdl})}$$

In [ ]:
class SparseRetriever:
    """
    Sparse retriever using bm25s — 50-100x faster than rank_bm25 via scipy
    sparse matrix indexing (only scores docs containing query terms).
    """
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1        = k1
        self.b         = b
        self.doc_ids   = []
        self.retriever = None
        self._built    = False

    def build_index(self, doc_ids: List[str], tokenized_corpus: List[List[str]]) -> None:
        logger.info(f"Building bm25s index (k1={self.k1}, b={self.b}) over {len(doc_ids):,} docs...")
        t0 = time.perf_counter()
        self.doc_ids   = doc_ids

        # bm25s.tokenize() expects raw strings
        corpus_strings = [" ".join(tokens) for tokens in tokenized_corpus]
        corpus_tokens  = bm25s.tokenize(corpus_strings, show_progress=False)

        self.retriever = bm25s.BM25(k1=self.k1, b=self.b, corpus=doc_ids)
        self.retriever.index(corpus_tokens)
        self._built = True
        logger.info(f"bm25s index built in {time.perf_counter()-t0:.2f}s")

    def search(self, query_tokens: List[str], top_k: int = 100) -> List[Tuple[str, float]]:
        if not self._built:
            raise RuntimeError("Index not built.")
        k = min(top_k, len(self.doc_ids))

        # Tokenize query as a single raw string
        query_str = " ".join(query_tokens)
        q_tokens  = bm25s.tokenize([query_str], show_progress=False)

        results, scores = self.retriever.retrieve(q_tokens, k=k)

        return [
            (str(doc_id), float(score))
            for doc_id, score in zip(results[0], scores[0])
            if float(score) > 0
        ]

    def evaluate(self, queries, qrels, processor, evaluator, top_k=100):
        all_results = {}
        for qid, qtext in tqdm(queries.items(), desc='BM25 search'):
            qtoks = processor.tokenize(qtext)
            all_results[qid] = self.search(qtoks, top_k=top_k)
        return evaluator.evaluate_run(all_results, qrels)

    def measure_latency(self, queries, processor, n_queries=50):
        sample = list(queries.items())[:n_queries]
        self.search(processor.tokenize(sample[0][1]), top_k=10)  # warmup
        latencies = []
        for _, qtext in sample:
            t0 = time.perf_counter()
            self.search(processor.tokenize(qtext), top_k=TOP_K)
            latencies.append((time.perf_counter()-t0)*1000)
        return {
            'mean_ms'  : round(np.mean(latencies),   2),
            'median_ms': round(np.median(latencies),  2),
            'p95_ms'   : round(np.percentile(latencies, 95), 2),
        }

sparse_retriever = SparseRetriever(k1=BM25_K1, b=BM25_B)
sparse_retriever.build_index(doc_ids, tokenized_corpus)
print("bm25s index ready.")

---
## Section 8 — Dense Retriever (BGE + FAISS HNSW)

In [ ]:
class DenseRetriever:
    """
    BGE dense retriever with FAISS inner-product index.
    """
    _EMB_FILE   = 'embeddings.npy'
    _IDS_FILE   = 'doc_ids.pkl'
    _META_FILE  = 'metadata.pkl'

    def __init__(self, model_name=SBERT_MODEL, batch_size=BATCH_SIZE, device=DEVICE):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device     = device
        self.model      = None
        self.faiss_index = None
        self.doc_ids    = []
        self.embeddings = None
        self._built     = False
        self._use_instruction = any(p in model_name for p in ('bge-', 'BAAI/bge', 'e5-', 'intfloat/e5'))

    def _load_model(self):
        if self.model is None:
            logger.info(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name, device=self.device)
            if 'cuda' in self.device:
                self.model.half()
            self.model.max_seq_length = 512

    def generate_embeddings(self, texts, is_query=False):
        self._load_model()
        if is_query and self._use_instruction:
            texts = [BGE_QUERY_INSTRUCTION + t for t in texts]
        embeddings = self.model.encode(
            texts, batch_size=self.batch_size, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        )
        return embeddings.astype(np.float32)

    def build_index(self, doc_ids, raw_texts, cache_dir=CACHE_DIR, chunk_to_doc=None):
        emb_path = os.path.join(cache_dir, self._EMB_FILE)
        ids_path = os.path.join(cache_dir, self._IDS_FILE)
        meta_path = os.path.join(cache_dir, self._META_FILE)

        if os.path.exists(emb_path) and os.path.exists(ids_path) and self._cache_valid(meta_path):
            logger.info("Cache hit — loading embeddings from disk.")
            self.embeddings = np.load(emb_path)
            with open(ids_path, 'rb') as f:
                self.doc_ids = pickle.load(f)
        else:
            logger.info(f"Generating embeddings for {len(raw_texts):,} chunks...")
            self.doc_ids = chunk_to_doc if chunk_to_doc is not None else doc_ids
            self.embeddings = self.generate_embeddings(raw_texts, is_query=False)
            np.save(emb_path, self.embeddings)
            with open(ids_path, 'wb') as f:
                pickle.dump(self.doc_ids, f)
            with open(meta_path, 'wb') as f:
                pickle.dump({'model_name': self.model_name, 'num_chunks': len(self.doc_ids),
                             'dim': self.embeddings.shape[1]}, f)

        self._build_faiss(index_type='hnsw')
        self._built = True
        logger.info(f"Dense retriever ready — {len(self.doc_ids):,} chunks, dim={self.embeddings.shape[1]}")

    def _cache_valid(self, meta_path):
        if not os.path.exists(meta_path):
            return False
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        return meta.get('model_name') == self.model_name

    def _build_faiss(self, index_type='flat'):
        dim = self.embeddings.shape[1]
        if index_type == 'hnsw':
            index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
            index.hnsw.efConstruction = 200
            index.hnsw.efSearch = 128
            index.add(self.embeddings)
            self.faiss_index = index
        else:
            self.faiss_index = faiss.IndexFlatIP(dim)
            self.faiss_index.add(self.embeddings)
        assert self.faiss_index.metric_type == faiss.METRIC_INNER_PRODUCT

    def search(self, query_text, top_k=100):
        if not self._built:
            raise RuntimeError("Index not built.")
        query_input = (BGE_QUERY_INSTRUCTION + query_text) if self._use_instruction else query_text
        self._load_model()
        q_emb = self.model.encode(
            [query_input], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
        ).astype(np.float32)

        fetch_n = min(top_k * 5, self.faiss_index.ntotal)
        scores, indices = self.faiss_index.search(q_emb, fetch_n)

        doc_scores = {}
        for r, i in enumerate(indices[0]):
            if i < 0:
                continue
            doc_id = self.doc_ids[i]
            s = float(scores[0][r])
            if doc_id not in doc_scores or s > doc_scores[doc_id]:
                doc_scores[doc_id] = s
        return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def evaluate(self, queries, qrels, evaluator, top_k=100):
        self._load_model()
        qids = list(queries.keys())
        qtexts = [queries[qid] for qid in qids]
        if self._use_instruction:
            qtexts = [BGE_QUERY_INSTRUCTION + t for t in qtexts]

        q_embs = self.model.encode(
            qtexts, batch_size=64, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=True
        ).astype(np.float32)

        fetch_n = min(top_k * 5, self.faiss_index.ntotal)
        scores, indices = self.faiss_index.search(q_embs, fetch_n)

        all_results = {}
        for idx, qid in enumerate(qids):
            doc_scores = {}
            for r, i in enumerate(indices[idx]):
                if i < 0:
                    continue
                doc_id = self.doc_ids[i]
                s = float(scores[idx][r])
                if doc_id not in doc_scores or s > doc_scores[doc_id]:
                    doc_scores[doc_id] = s
            all_results[qid] = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        return evaluator.evaluate_run(all_results, qrels)

    def measure_latency(self, queries, n_queries=50):
        self._load_model()
        self.search(list(queries.values())[0], top_k=10)
        sample = list(queries.values())[:n_queries]
        latencies = []
        for qtext in sample:
            t0 = time.perf_counter()
            self.search(qtext, top_k=TOP_K)
            latencies.append((time.perf_counter()-t0)*1000)
        return {'mean_ms': round(np.mean(latencies),2), 'median_ms': round(np.median(latencies),2),
                'p95_ms': round(np.percentile(latencies,95),2)}

print("DenseRetriever class defined.")

In [ ]:
# ── Clear cache and build fresh ───────────────────────────────────────────────
for _f in ['embeddings.npy', 'doc_ids.pkl', 'metadata.pkl']:
    _p = os.path.join(CACHE_DIR, _f)
    if os.path.exists(_p):
        os.remove(_p)

dense_retriever = DenseRetriever(model_name=SBERT_MODEL, batch_size=BATCH_SIZE, device=DEVICE)
dense_retriever.build_index(
    doc_ids=chunk_to_doc, raw_texts=chunk_texts,
    cache_dir=CACHE_DIR, chunk_to_doc=chunk_to_doc,
)

print(f"FAISS: HNSW index, efSearch={dense_retriever.faiss_index.hnsw.efSearch}")
print(f"Embedding dim: {dense_retriever.embeddings.shape[1]}")
print(f"Metric type: {'INNER_PRODUCT' if dense_retriever.faiss_index.metric_type == faiss.METRIC_INNER_PRODUCT else 'L2'}")

---
## Section 9 — Dense Retrieval Sanity Check

In [ ]:
# ── Embedding health check ────────────────────────────────────────────────────
emb = dense_retriever.embeddings
print("=== Embedding Health ===")
print(f"dtype : {emb.dtype}")
print(f"shape : {emb.shape}")
print(f"NaN   : {np.isnan(emb).any()}")
print(f"Inf   : {np.isinf(emb).any()}")
print(f"norms : {np.linalg.norm(emb[:5], axis=1)}")

# ── End-to-end single-query sanity check ─────────────────────────────────────
qid = list(queries_raw.keys())[0]
qtext = queries_raw[qid]
gold_eid = list(qrels[qid].keys())[0]

print(f"\nQuery: {qtext}")
print(f"Gold doc: {gold_eid}")
print(f"Gold in index: {gold_eid in dense_retriever.doc_ids}")

gold_chunk_idxs = [i for i, d in enumerate(dense_retriever.doc_ids) if d == gold_eid]
print(f"Gold doc chunks in index: {len(gold_chunk_idxs)}")

q_input = (BGE_QUERY_INSTRUCTION + qtext) if dense_retriever._use_instruction else qtext
q_emb = dense_retriever.model.encode([q_input], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

for ci in gold_chunk_idxs[:3]:
    sim = float(np.dot(q_emb[0], dense_retriever.embeddings[ci]))
    print(f"  chunk {ci}: cosine sim = {sim:.4f}")

rand_idx = random.randrange(len(chunk_texts))
rand_sim = float(np.dot(q_emb[0], dense_retriever.embeddings[rand_idx]))
print(f"Random chunk sim = {rand_sim:.4f}")

# Verify FAISS ranking
scores, indices = dense_retriever.faiss_index.search(q_emb, min(500, dense_retriever.faiss_index.ntotal))
for ci in gold_chunk_idxs[:1]:
    if ci in indices[0]:
        rank = np.where(indices[0] == ci)[0][0]
        print(f"\nGold chunk rank in FAISS: {rank} (should be low)")
print(f"Top-3 FAISS doc_ids: {[dense_retriever.doc_ids[i] for i in indices[0][:3]]}")

---
## Section 10 — Evaluator

In [ ]:
class Evaluator:
    """Computes IR evaluation metrics: Recall@k, Precision@k, MRR, NDCG@k."""

    def __init__(self, k_values=EVAL_K_VALUES):
        self.k_values = sorted(k_values)

    @staticmethod
    def recall_at_k(results, relevant, k):
        if not relevant: return 0.0
        return len({d for d,_ in results[:k]} & relevant) / len(relevant)

    @staticmethod
    def precision_at_k(results, relevant, k):
        if k == 0: return 0.0
        return sum(1 for d,_ in results[:k] if d in relevant) / k

    @staticmethod
    def mrr(results, relevant):
        for rank, (d,_) in enumerate(results, 1):
            if d in relevant:
                return 1.0 / rank
        return 0.0

    @staticmethod
    def ndcg_at_k(results, relevant, k):
        dcg = sum(1.0/math.log2(r+2) for r,(d,_) in enumerate(results[:k]) if d in relevant)
        idcg = sum(1.0/math.log2(i+2) for i in range(min(len(relevant), k)))
        return dcg/idcg if idcg > 0 else 0.0

    def evaluate_query(self, results, relevant):
        m = {'mrr': self.mrr(results, relevant)}
        for k in self.k_values:
            m[f'recall@{k}'] = self.recall_at_k(results, relevant, k)
            m[f'precision@{k}'] = self.precision_at_k(results, relevant, k)
            m[f'ndcg@{k}'] = self.ndcg_at_k(results, relevant, k)
        return m

    def evaluate_run(self, all_results, qrels):
        per_q = {}
        for qid, results in all_results.items():
            per_q[qid] = self.evaluate_query(results, set(qrels.get(qid, {}).keys()))
        agg = defaultdict(list)
        for metrics in per_q.values():
            for m, v in metrics.items():
                agg[m].append(v)
        return {m: round(float(np.mean(v)),4) for m,v in agg.items()}, per_q

evaluator = Evaluator()
print("Evaluator ready.")

---
## Section 11 — RRF Fusion, Jina Reranker & Hybrid Retriever

Replacing the CrossEncoder with `jina-reranker-v3` (0.6B listwise model).
The wrapper supports batching documents (default batch size of 32 docs) to fit within context limits.
It loads the model as a sequence classification model (leveraging its pointwise logits fallback).

In [ ]:
class ReciprocalRankFusion:
    """Reciprocal Rank Fusion (Cormack et al., 2009)."""
    def __init__(self, k=60.0, weights=None):
        self.k = k
        self.weights = weights

    def fuse(self, ranked_lists):
        weights = self.weights or [1.0] * len(ranked_lists)
        doc_scores = {}
        for w, ranked in zip(weights, ranked_lists):
            for rank, (doc_id, _) in enumerate(ranked, start=1):
                doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + w / (self.k + rank)
        return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)


class JinaReranker:
    """jina-reranker-v3 wrapper matching the original CrossEncoderReranker interface.

    The model is a listwise LBNL reranker. It scores a query against a list of documents
    in shared context when possible, and this wrapper batches candidates in groups of
    JINA_BATCH_DOCS to stay below context and memory limits.
    """
    def __init__(self, model_name=JINA_RERANKER_MODEL, device=None, max_length=JINA_MAX_LENGTH, batch_size=JINA_BATCH_DOCS):
        self.model_name = model_name
        self.device = device or DEVICE
        self.max_length = max_length
        self.batch_size = min(batch_size, 64)
        self._model = None
        self._tokenizer = None
        self._corpus = {}
        self._ready = False

    def _load_model(self):
        if self._model is None:
            try:
                self._tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
                self._model = AutoModelForSequenceClassification.from_pretrained(
                    self.model_name,
                    torch_dtype=torch.float16 if "cuda" in self.device else torch.float32,
                    trust_remote_code=True,
                )
                self._model.to(self.device)
                self._model.eval()
            except Exception as exc:
                raise RuntimeError(
                    f"Could not load {self.model_name}. Check HuggingFace access/network and that "
                    "trust_remote_code=True is allowed."
                ) from exc

    def build_index(self, doc_ids, corpus_raw, chunk_texts=None, chunk_to_doc=None):
        if chunk_texts is not None and chunk_to_doc is not None:
            doc_chunks = defaultdict(list)
            for ct, cd in zip(chunk_texts, chunk_to_doc):
                doc_chunks[cd].append(ct)
            for doc_id in doc_ids:
                self._corpus[doc_id] = ' '.join(doc_chunks.get(doc_id, [""]))
        else:
            for doc_id in doc_ids:
                doc = corpus_raw.get(doc_id, {})
                self._corpus[doc_id] = (doc.get('title','') + ' ' + doc.get('text','')).strip()
        self._load_model()
        self._ready = True
        logger.info(f"Jina reranker ready: {self.model_name} on {self.device}")

    def _normalize_scores(self, raw, n_docs):
        if isinstance(raw, torch.Tensor):
            raw = raw.detach().cpu().flatten().tolist()
        if isinstance(raw, list) and raw and isinstance(raw[0], dict):
            scores = [0.0] * n_docs
            for item in raw:
                idx = int(item.get("index", len(scores)))
                if 0 <= idx < n_docs:
                    scores[idx] = float(item.get("relevance_score", item.get("score", 0.0)))
            return scores
        if isinstance(raw, list):
            return [float(x) for x in raw[:n_docs]]
        return [float(raw)] * n_docs

    def _score_batch(self, query, documents):
        self._load_model()
        if hasattr(self._model, "rerank"):
            raw = self._model.rerank(query=query, documents=documents, max_length=self.max_length)
            return self._normalize_scores(raw, len(documents))
        prompts = [
            f"<|im_start|>system\nYou are a search relevance expert.\n<|im_end|>\n"
            f"<|im_start|>user\nQuery: {query}\nDocument: {doc}\n<|im_end|>"
            for doc in documents
        ]
        enc = self._tokenizer(prompts, max_length=self.max_length, truncation=True, padding=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self._model(**enc)
            logits = out.logits.squeeze(-1)
        return logits.detach().float().cpu().tolist()

    def _score_single(self, query, document):
        return float(self._score_batch(query, [document])[0])

    def rerank(self, query, candidates, top_n=RERANKER_TOP_N):
        if not self._ready or not candidates:
            return candidates
        target = candidates[:top_n]
        remaining = candidates[top_n:]
        scored = []
        for start in range(0, len(target), self.batch_size):
            batch = target[start:start + self.batch_size]
            docs = [self._corpus.get(doc_id, "") for doc_id, _ in batch]
            scores = self._score_batch(query, docs)
            scored.extend((doc_id, float(score)) for (doc_id, _), score in zip(batch, scores))
        reranked = sorted(scored, key=lambda x: x[1], reverse=True)
        assert {d for d, _ in reranked} == {d for d, _ in target}, "Reranker changed candidate doc ids"
        return reranked + list(remaining)


class HybridRetriever:
    def __init__(self, sparse, dense, rrf, processor, reranker=None,
                 fetch_k=FETCH_K, bm25_weight=BM25_WEIGHT, dense_weight=DENSE_WEIGHT):
        self.sparse = sparse
        self.dense = dense
        self.rrf = rrf
        self.processor = processor
        self.reranker = reranker
        self.fetch_k = fetch_k
        self.rrf.weights = [bm25_weight, dense_weight]
        self._tok_cache = {}

    def retrieve(self, query, top_k=100):
        qtoks = self._tok_cache.get(query)
        if qtoks is None:
            qtoks = self.processor.tokenize(query)
            self._tok_cache[query] = qtoks

        sparse_res = self.sparse.search(qtoks, self.fetch_k)
        dense_res = self.dense.search(query, self.fetch_k)
        fused = self.rrf.fuse([sparse_res, dense_res])

        if self.reranker is not None:
            fused = self.reranker.rerank(query, fused, top_n=RERANKER_TOP_N)
        return fused[:top_k]

    def evaluate(self, queries, qrels, evaluator, top_k=100):
        all_results = {}
        desc = 'Hybrid + Rerank' if self.reranker else 'Hybrid'
        for qid in tqdm(queries.keys(), desc=desc):
            all_results[qid] = self.retrieve(queries[qid], top_k=top_k)
        return evaluator.evaluate_run(all_results, qrels)

print("RRF, JinaReranker, HybridRetriever classes defined.")

---
## Section 12 — Baseline Evaluation

In [ ]:
# ── Instantiate default hybrid (no reranker) ──────────────────────────────────
rrf = ReciprocalRankFusion(k=RRF_K)
hybrid_retriever = HybridRetriever(
    sparse=sparse_retriever, dense=dense_retriever, rrf=rrf,
    processor=processor, fetch_k=FETCH_K,
)

# ── Run baselines ─────────────────────────────────────────────────────────────
print("Running Baseline BM25...")
bm25_agg, bm25_per_q = sparse_retriever.evaluate(queries_raw, qrels, processor, evaluator, top_k=TOP_K)

print("\nRunning Baseline Dense (BGE)...")
dense_agg, dense_per_q = dense_retriever.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)

print("\nRunning Baseline Hybrid (RRF)...")
hybrid_agg, hybrid_per_q = hybrid_retriever.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)

# ── Comparison table ──────────────────────────────────────────────────────────
KEY_METRICS = ['mrr', 'ndcg@10', 'ndcg@100', 'recall@10', 'recall@100', 'precision@10']
rows = []
for name, agg in [('BM25', bm25_agg), ('Dense BGE', dense_agg), ('Hybrid RRF', hybrid_agg)]:
    rows.append({'System': name, **{m: round(agg.get(m,0),4) for m in KEY_METRICS}})
baseline_df = pd.DataFrame(rows).set_index('System')
print("\n=== Baseline Retrieval Comparison ===")
print(baseline_df.to_string())

In [ ]:
# ── Baseline visualization ────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric in zip(axes, KEY_METRICS):
    vals = [rows[i].get(metric, 0) for i in range(3)]
    bars = ax.bar(['BM25', 'Dense BGE', 'Hybrid RRF'], vals, color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%.4f', padding=2, fontsize=9)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Baseline System Comparison', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 13 — Parameter Optimization

In [ ]:
# BM25 parameter sweep
bm25_configs = [
    (0.9, 0.4), (1.2, 0.5), (1.2, 0.75), (1.5, 0.5), (1.5, 0.75),
    (1.8, 0.6), (1.8, 0.8), (2.0, 0.75), (2.0, 0.9),
]

bm25_results = []
for k1, b in bm25_configs:
    print(f"  k1={k1}, b={b} ...", end=' ')
    sr = SparseRetriever(k1=k1, b=b)
    sr.build_index(doc_ids, tokenized_corpus)
    agg, _ = sr.evaluate(queries_raw, qrels, processor, evaluator, top_k=TOP_K)
    row = {'k1': k1, 'b': b, **{m: round(agg[m],4) for m in ['mrr','ndcg@10','ndcg@100','recall@10','recall@100']}}
    bm25_results.append(row)
    print(f"NDCG@10={row['ndcg@10']:.4f}")

bm25_df = pd.DataFrame(bm25_results)
print("\n=== BM25 Hyperparameter Search ===")
print(bm25_df.to_string(index=False))

best = bm25_df.loc[bm25_df['ndcg@10'].idxmax()]
BEST_BM25_K1, BEST_BM25_B = float(best['k1']), float(best['b'])
print(f"\n>>> Best BM25: k1={BEST_BM25_K1}, b={BEST_BM25_B} (NDCG@10={best['ndcg@10']:.4f})")

# Rebuild BM25 with optimal params
sparse_retriever = SparseRetriever(k1=BEST_BM25_K1, b=BEST_BM25_B)
sparse_retriever.build_index(doc_ids, tokenized_corpus)
print(f"BM25 rebuilt with k1={BEST_BM25_K1}, b={BEST_BM25_B}")

In [ ]:
# RRF constant k sweep
rrf_k_values = [3, 5, 10, 20, 40, 60, 80, 100]
rrf_results = []

for k_val in rrf_k_values:
    print(f"  RRF k={k_val} ...", end=' ')
    rrf_tmp = ReciprocalRankFusion(k=k_val)
    hr = HybridRetriever(sparse=sparse_retriever, dense=dense_retriever, rrf=rrf_tmp,
                          processor=processor, fetch_k=FETCH_K)
    agg, _ = hr.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)
    row = {'rrf_k': k_val, **{m: round(agg[m],4) for m in ['mrr','ndcg@10','ndcg@100','recall@10','recall@100']}}
    rrf_results.append(row)
    print(f"NDCG@10={row['ndcg@10']:.4f}")

rrf_df = pd.DataFrame(rrf_results)
print("\n=== RRF k Search ===")
print(rrf_df.to_string(index=False))

best_rrf = rrf_df.loc[rrf_df['ndcg@10'].idxmax()]
BEST_RRF_K = int(best_rrf['rrf_k'])
print(f"\n>>> Best RRF k={BEST_RRF_K} (NDCG@10={best_rrf['ndcg@10']:.4f})")

In [ ]:
# Weighted RRF sweep
weight_configs = [
    (0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.55, 0.45),
    (0.6, 0.4), (0.65, 0.35), (0.7, 0.3), (0.75, 0.25),
    (0.8, 0.2), (0.9, 0.1),
]

weight_results = []
for bw, dw in weight_configs:
    print(f"  BM25_w={bw}, Dense_w={dw} ...", end=' ')
    rrf_tmp = ReciprocalRankFusion(k=BEST_RRF_K, weights=[bw, dw])
    hr = HybridRetriever(sparse=sparse_retriever, dense=dense_retriever, rrf=rrf_tmp,
                          processor=processor, fetch_k=FETCH_K, bm25_weight=bw, dense_weight=dw)
    agg, _ = hr.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)
    row = {'bm25_w': bw, 'dense_w': dw,
           **{m: round(agg[m],4) for m in ['mrr','ndcg@10','ndcg@100','recall@10','recall@100','precision@10']}}
    weight_results.append(row)
    print(f"NDCG@10={row['ndcg@10']:.4f}")

weight_df = pd.DataFrame(weight_results)
print("\n=== Weighted RRF Results ===")
print(weight_df.to_string(index=False))

best_w = weight_df.loc[weight_df['ndcg@10'].idxmax()]
BEST_BM25_W, BEST_DENSE_W = float(best_w['bm25_w']), float(best_w['dense_w'])
print(f"\n>>> Best weights: BM25={BEST_BM25_W}, Dense={BEST_DENSE_W} (NDCG@10={best_w['ndcg@10']:.4f})")

In [ ]:
# Candidate pool size sweep
pool_sizes = [50, 100, 200, 300, 500]
pool_results = []

for pool in pool_sizes:
    print(f"  Pool={pool} ...", end=' ')
    rrf_tmp = ReciprocalRankFusion(k=BEST_RRF_K, weights=[BEST_BM25_W, BEST_DENSE_W])
    hr = HybridRetriever(sparse=sparse_retriever, dense=dense_retriever, rrf=rrf_tmp,
                          processor=processor, fetch_k=pool, bm25_weight=BEST_BM25_W, dense_weight=BEST_DENSE_W)
    t0 = time.perf_counter()
    agg, _ = hr.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)
    latency = (time.perf_counter()-t0)/len(queries_raw)*1000
    row = {'pool': pool, 'latency_ms': round(latency,2),
           **{m: round(agg[m],4) for m in ['mrr','ndcg@10','ndcg@100','recall@10','recall@100']}}
    pool_results.append(row)
    print(f"NDCG@10={row['ndcg@10']:.4f}  Recall@100={row['recall@100']:.4f}")

pool_df = pd.DataFrame(pool_results)
print("\n=== Pool Size Results ===")
print(pool_df.to_string(index=False))

best_pool = pool_df.loc[pool_df['ndcg@10'].idxmax()]
BEST_POOL = int(best_pool['pool'])
print(f"\n>>> Best pool: {BEST_POOL} (NDCG@10={best_pool['ndcg@10']:.4f})")

---
## Section 17 — Jina Reranker v3 (Off-the-Shelf)

In [ ]:
# ── Build off-the-shelf Jina Reranker ──────────────────────────────────────────
jina_reranker_offshelf = JinaReranker(
    model_name=JINA_RERANKER_MODEL,
    device=DEVICE,
    max_length=JINA_MAX_LENGTH,
    batch_size=JINA_BATCH_DOCS,
)
jina_reranker_offshelf.build_index(doc_ids=doc_ids, corpus_raw=corpus_raw,
                                   chunk_texts=chunk_texts, chunk_to_doc=chunk_to_doc)

# Sanity check
s_high = jina_reranker_offshelf._score_single(
    "What contracts did Enron sign with California utilities?",
    "Enron Corporation entered into long-term energy contracts with Pacific Gas and Electric."
)
s_low = jina_reranker_offshelf._score_single(
    "What contracts did Enron sign with California utilities?",
    "The weather forecast for tomorrow shows sunny skies and mild temperatures."
)
print(f"Reranker sanity: relevant={s_high:.4f}, irrelevant={s_low:.4f}")
assert s_high > s_low, f"Reranker sanity check failed: {s_high:.4f} vs {s_low:.4f}"

# ── Build hybrid + reranker ──────────────────────────────────────────────────
rrf_opt = ReciprocalRankFusion(k=BEST_RRF_K, weights=[BEST_BM25_W, BEST_DENSE_W])
hybrid_jina_offshelf = HybridRetriever(
    sparse=sparse_retriever, dense=dense_retriever, rrf=rrf_opt,
    processor=processor, reranker=jina_reranker_offshelf,
    fetch_k=BEST_POOL, bm25_weight=BEST_BM25_W, dense_weight=BEST_DENSE_W,
)

print("\nEvaluating Hybrid + Jina Reranker v3 (off-the-shelf)...")
offshelf_agg, offshelf_per_q = hybrid_jina_offshelf.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)
print("\n=== Hybrid + Jina Reranker v3 (off-the-shelf) ===")
for m, v in sorted(offshelf_agg.items()):
    print(f"  {m:<20}: {v:.4f}")

---
## Section 18 — Standalone Setup for Fine-Tuning & Evaluation

Run this cell (after pip installs and imports) to skip Sections 3–17 and jump directly to fine-tuning.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STANDALONE SETUP — Jump directly to fine-tuning (Section 19).
# ════════════════════════════════════════════════════════════════════════════════
import random
import hashlib

# ── 1. Dataset loading ────────────────────────────────────────────────────────
def load_enronqa(split_ratio: float = 0.2, seed: int = 42, max_queries: int = 500):
    ds = load_dataset("MichaelR207/enron_qa_0922", split="train")
    corpus_raw = {}
    for row in ds:
        eid = row["path"]
        email_text = row["email"] or ""
        subject = ""
        for line in email_text.splitlines():
            if line.startswith("Subject:"):
                subject = line.replace("Subject:", "").strip()
                break
        corpus_raw[eid] = {"title": subject, "text": email_text}
    all_eids = list(corpus_raw.keys())
    test_eids = set()
    for eid in all_eids:
        bucket = int(hashlib.md5(eid.encode()).hexdigest(), 16) % 100
        if bucket < int(split_ratio * 100):
            test_eids.add(eid)
    queries_raw, qrels = {}, {}
    q_counter = 0
    for row in ds:
        if q_counter >= max_queries:
            break
        eid = row["path"]
        if eid not in test_eids:
            continue
        for q_text in (row.get("questions") or []):
            if q_counter >= max_queries:
                break
            if not q_text or len(q_text.split()) < 3:
                continue
            qid = f"q_{q_counter:06d}"
            queries_raw[qid] = q_text
            qrels[qid] = {eid: 1}
            q_counter += 1
    train_pairs = []
    for row in ds:
        eid = row["path"]
        if eid in test_eids:
            continue
        for q_text in (row.get("questions") or []):
            if not q_text or len(q_text.split()) < 3:
                continue
            train_pairs.append({"query": q_text, "doc_id": eid})
    return corpus_raw, queries_raw, qrels, train_pairs

corpus_raw, queries_raw, qrels, train_pairs_raw = load_enronqa()

# ── 2. Email preprocessor ─────────────────────────────────────────────────────
class EnronEmailProcessor:
    _PREAMBLE_SPLIT = re.compile(r'={5,}\s*\n')
    _FWD_PATTERN    = re.compile(r'-{3,}\s*(?:Forwarded|Original Message)\s*-{3,}', re.IGNORECASE)
    _HEADER_PATTERN = re.compile(r'^(From|To|Cc|Sent|Date|Subject):\s*(.+)', re.IGNORECASE | re.MULTILINE)
    _EMAIL_PATTERN  = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+')

    def __init__(self, remove_stopwords=True, min_token_len=2):
        self.remove_stopwords = remove_stopwords
        self.min_token_len = min_token_len
        self._stopwords = set(stopwords.words('english')) if remove_stopwords else set()

    def preprocess_document(self, text):
        if not text:
            return ""
        emails = self._EMAIL_PATTERN.findall(text)
        email_parts = []
        for email in emails:
            local, domain = email.split('@', 1)
            email_parts.extend([email.replace('@', ' at '), local.replace('.', ' ')])
        text = re.sub(r'[^\w\s\-]', ' ', text)
        tokens = text.split()
        filtered = []
        for tok in tokens:
            tok_lower = tok.lower()
            if re.fullmatch(r'\d+', tok_lower):
                continue
            if self.remove_stopwords and tok_lower in self._stopwords:
                continue
            if len(tok_lower) < self.min_token_len:
                continue
            filtered.append(tok_lower)
        for part in email_parts:
            for tok in part.lower().split():
                if len(tok) >= self.min_token_len and tok not in self._stopwords:
                    filtered.append(tok)
        return ' '.join(filtered)

    def tokenize(self, text):
        pp = self.preprocess_document(text)
        return pp.split() if pp else []

    def clean_email_body(self, email_text):
        if not email_text:
            return ""
        parts = self._PREAMBLE_SPLIT.split(email_text, maxsplit=1)
        if len(parts) == 2:
            body = parts[1].strip()
            if len(body.split()) >= 5:
                return body
        return email_text.strip()

    def parse_email_fields(self, email_text):
        fields = {'from': '', 'to': '', 'cc': '', 'subject': '', 'body': ''}
        body = self.clean_email_body(email_text)
        for match in self._HEADER_PATTERN.finditer(body[:2000]):
            key = match.group(1).lower()
            if key in ('sent', 'date'):
                continue
            if key in fields:
                fields[key] += ' ' + match.group(2).strip()
        fields['body'] = body
        return fields

    def combine_fields_bm25(self, doc):
        subject = doc.get('title', '') or ''
        raw = doc.get('text', '') or ''
        fields = self.parse_email_fields(raw)
        parts = [subject] * 3
        if fields['from'].strip():
            parts.extend([fields['from']] * 2)
        if fields['to'].strip():
            parts.extend([fields['to']] * 2)
        parts.append(fields['body'])
        return ' '.join(parts).strip()

    def combine_fields_dense(self, doc):
        subject = doc.get('title', '') or ''
        raw = doc.get('text', '') or ''
        body = self.clean_email_body(raw)
        if len(body.split()) < 5:
            body = raw
        return f"{subject}\n{body}".strip()

    def chunk_text(self, text, chunk_size=200, overlap=50):
        segments = self._FWD_PATTERN.split(text)
        all_chunks = []
        for segment in segments:
            segment = segment.strip()
            if not segment:
                continue
            words = segment.split()
            if len(words) <= chunk_size:
                if words:
                    all_chunks.append(segment)
            else:
                start = 0
                while start < len(words):
                    end = start + chunk_size
                    all_chunks.append(' '.join(words[start:end]))
                    if end >= len(words):
                        break
                    start += chunk_size - overlap
        return all_chunks if all_chunks else ([text] if text.strip() else [])

    def preprocess_corpus(self, corpus):
        doc_ids, tokenized_corpus = [], []
        chunk_texts, chunk_to_doc = [], []
        for doc_id, doc in tqdm(corpus.items(), desc='Preprocessing corpus'):
            combined = self.combine_fields_bm25(doc)
            tokens = self.tokenize(combined)
            doc_ids.append(doc_id)
            tokenized_corpus.append(tokens)
            dense_text = self.combine_fields_dense(doc)
            chunks = self.chunk_text(dense_text, chunk_size=DENSE_CHUNK_SIZE, overlap=DENSE_CHUNK_OVERLAP)
            if not chunks:
                chunks = [doc.get('title', '') or '']
            for c in chunks:
                chunk_texts.append(c)
                chunk_to_doc.append(doc_id)
        return doc_ids, tokenized_corpus, chunk_texts, chunk_to_doc

processor = EnronEmailProcessor(remove_stopwords=True, min_token_len=2)
doc_ids, tokenized_corpus, chunk_texts, chunk_to_doc = processor.preprocess_corpus(corpus_raw)

# ── 3. Sparse retriever (bm25s) ───────────────────────────────────────────────
class SparseRetriever:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b  = b
        self.doc_ids   = []
        self.retriever = None
        self._built    = False

    def build_index(self, doc_ids, tokenized_corpus):
        logger.info(f"Building bm25s index (k1={self.k1}, b={self.b}) over {len(doc_ids):,} docs...")
        t0 = time.perf_counter()
        self.doc_ids = doc_ids
        corpus_strings = [" ".join(tokens) for tokens in tokenized_corpus]
        corpus_tokens  = bm25s.tokenize(corpus_strings, show_progress=False)
        self.retriever = bm25s.BM25(k1=self.k1, b=self.b, corpus=doc_ids)
        self.retriever.index(corpus_tokens)
        self._built = True
        logger.info(f"bm25s index built in {time.perf_counter()-t0:.2f}s")

    def search(self, query_tokens, top_k=100):
        if not self._built:
            raise RuntimeError("Index not built.")
        k = min(top_k, len(self.doc_ids))
        query_str = " ".join(query_tokens)
        q_tokens  = bm25s.tokenize([query_str], show_progress=False)
        results, scores = self.retriever.retrieve(q_tokens, k=k)
        return [(str(doc_id), float(score))
                for doc_id, score in zip(results[0], scores[0])
                if float(score) > 0]

    def evaluate(self, queries, qrels, processor, evaluator, top_k=100):
        all_results = {}
        for qid, qtext in tqdm(queries.items(), desc='BM25 search'):
            all_results[qid] = self.search(processor.tokenize(qtext), top_k=top_k)
        return evaluator.evaluate_run(all_results, qrels)

    def measure_latency(self, queries, processor, n_queries=50):
        sample = list(queries.items())[:n_queries]
        self.search(processor.tokenize(sample[0][1]), top_k=10)
        latencies = []
        for _, qtext in sample:
            t0 = time.perf_counter()
            self.search(processor.tokenize(qtext), top_k=TOP_K)
            latencies.append((time.perf_counter()-t0)*1000)
        return {'mean_ms': round(np.mean(latencies),2), 'median_ms': round(np.median(latencies),2),
                'p95_ms': round(np.percentile(latencies,95),2)}

sparse_retriever = SparseRetriever(k1=BM25_K1, b=BM25_B)
sparse_retriever.build_index(doc_ids, tokenized_corpus)

# ── 4. Dense retriever (BGE + FAISS HNSW) ────────────────────────────────────
class DenseRetriever:
    _EMB_FILE  = 'embeddings.npy'
    _IDS_FILE  = 'doc_ids.pkl'
    _META_FILE = 'metadata.pkl'

    def __init__(self, model_name=SBERT_MODEL, batch_size=BATCH_SIZE, device=DEVICE):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device     = device
        self.model = None; self.faiss_index = None; self.doc_ids = []
        self.embeddings = None; self._built = False
        self._use_instruction = any(p in model_name for p in ('bge-', 'BAAI/bge', 'e5-', 'intfloat/e5'))

    def _load_model(self):
        if self.model is None:
            logger.info(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name, device=self.device)
            if 'cuda' in self.device:
                self.model.half()
            self.model.max_seq_length = 512

    def generate_embeddings(self, texts, is_query=False):
        self._load_model()
        if is_query and self._use_instruction:
            texts = [BGE_QUERY_INSTRUCTION + t for t in texts]
        return self.model.encode(texts, batch_size=self.batch_size, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

    def _cache_valid(self, meta_path):
        if not os.path.exists(meta_path):
            return False
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        return meta.get('model_name') == self.model_name

    def _build_faiss(self, index_type='flat'):
        dim = self.embeddings.shape[1]
        if index_type == 'hnsw':
            index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
            index.hnsw.efConstruction = 200
            index.hnsw.efSearch = 128
            index.add(self.embeddings)
            self.faiss_index = index
        else:
            self.faiss_index = faiss.IndexFlatIP(dim)
            self.faiss_index.add(self.embeddings)
        assert self.faiss_index.metric_type == faiss.METRIC_INNER_PRODUCT

    def build_index(self, doc_ids, raw_texts, cache_dir=CACHE_DIR, chunk_to_doc=None):
        emb_path  = os.path.join(cache_dir, self._EMB_FILE)
        ids_path  = os.path.join(cache_dir, self._IDS_FILE)
        meta_path = os.path.join(cache_dir, self._META_FILE)
        if os.path.exists(emb_path) and os.path.exists(ids_path) and self._cache_valid(meta_path):
            logger.info("Cache hit — loading embeddings from disk.")
            self.embeddings = np.load(emb_path)
            with open(ids_path, 'rb') as f:
                self.doc_ids = pickle.load(f)
        else:
            logger.info(f"Generating embeddings for {len(raw_texts):,} chunks...")
            self.doc_ids = chunk_to_doc if chunk_to_doc is not None else doc_ids
            self.embeddings = self.generate_embeddings(raw_texts, is_query=False)
            np.save(emb_path, self.embeddings)
            with open(ids_path, 'wb') as f:
                pickle.dump(self.doc_ids, f)
            with open(meta_path, 'wb') as f:
                pickle.dump({'model_name': self.model_name, 'num_chunks': len(self.doc_ids),
                             'dim': self.embeddings.shape[1]}, f)
        self._build_faiss(index_type='hnsw')
        self._built = True
        logger.info(f"Dense retriever ready — {len(self.doc_ids):,} chunks, dim={self.embeddings.shape[1]}")

    def search(self, query_text, top_k=100):
        if not self._built:
            raise RuntimeError("Index not built.")
        self._load_model()
        q_input = (BGE_QUERY_INSTRUCTION + query_text) if self._use_instruction else query_text
        q_emb = self.model.encode([q_input], convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
        fetch_n = min(top_k * 5, self.faiss_index.ntotal)
        scores, indices = self.faiss_index.search(q_emb, fetch_n)
        doc_scores = {}
        for r, i in enumerate(indices[0]):
            if i < 0:
                continue
            doc_id = self.doc_ids[i]
            s = float(scores[0][r])
            if doc_id not in doc_scores or s > doc_scores[doc_id]:
                doc_scores[doc_id] = s
        return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def evaluate(self, queries, qrels, evaluator, top_k=100):
        self._load_model()
        qids = list(queries.keys())
        qtexts = [queries[qid] for qid in qids]
        if self._use_instruction:
            qtexts = [BGE_QUERY_INSTRUCTION + t for t in qtexts]
        q_embs = self.model.encode(qtexts, batch_size=64, convert_to_numpy=True,
                                   normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
        fetch_n = min(top_k * 5, self.faiss_index.ntotal)
        scores, indices = self.faiss_index.search(q_embs, fetch_n)
        all_results = {}
        for idx, qid in enumerate(qids):
            doc_scores = {}
            for r, i in enumerate(indices[idx]):
                if i < 0:
                    continue
                doc_id = self.doc_ids[i]
                s = float(scores[idx][r])
                if doc_id not in doc_scores or s > doc_scores[doc_id]:
                    doc_scores[doc_id] = s
            all_results[qid] = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return evaluator.evaluate_run(all_results, qrels)

    def measure_latency(self, queries, n_queries=50):
        self._load_model()
        self.search(list(queries.values())[0], top_k=10)
        sample = list(queries.values())[:n_queries]
        latencies = []
        for qtext in sample:
            t0 = time.perf_counter()
            self.search(qtext, top_k=TOP_K)
            latencies.append((time.perf_counter()-t0)*1000)
        return {'mean_ms': round(np.mean(latencies),2), 'median_ms': round(np.median(latencies),2),
                'p95_ms': round(np.percentile(latencies,95),2)}

dense_retriever = DenseRetriever(model_name=SBERT_MODEL, batch_size=BATCH_SIZE, device=DEVICE)
dense_retriever.build_index(doc_ids=chunk_to_doc, raw_texts=chunk_texts,
                            cache_dir=CACHE_DIR, chunk_to_doc=chunk_to_doc)

# ── 5. Evaluator ──────────────────────────────────────────────────────────────
class Evaluator:
    def __init__(self, k_values=EVAL_K_VALUES):
        self.k_values = sorted(k_values)

    @staticmethod
    def recall_at_k(results, relevant, k):
        if not relevant: return 0.0
        return len({d for d,_ in results[:k]} & relevant) / len(relevant)

    @staticmethod
    def precision_at_k(results, relevant, k):
        if k == 0: return 0.0
        return sum(1 for d,_ in results[:k] if d in relevant) / k

    @staticmethod
    def mrr(results, relevant):
        for rank, (d,_) in enumerate(results, 1):
            if d in relevant:
                return 1.0 / rank
        return 0.0

    @staticmethod
    def ndcg_at_k(results, relevant, k):
        dcg  = sum(1.0/math.log2(r+2) for r,(d,_) in enumerate(results[:k]) if d in relevant)
        idcg = sum(1.0/math.log2(i+2) for i in range(min(len(relevant), k)))
        return dcg/idcg if idcg > 0 else 0.0

    def evaluate_query(self, results, relevant):
        m = {'mrr': self.mrr(results, relevant)}
        for k in self.k_values:
            m[f'ndcg@{k}']      = self.ndcg_at_k(results, relevant, k)
            m[f'recall@{k}']    = self.recall_at_k(results, relevant, k)
            m[f'precision@{k}'] = self.precision_at_k(results, relevant, k)
        return m

    def evaluate_run(self, all_results, qrels):
        per_query = {}
        for qid, results in all_results.items():
            relevant = set(qrels.get(qid, {}).keys())
            per_query[qid] = self.evaluate_query(results, relevant)
        keys = list(per_query[next(iter(per_query))].keys())
        agg = {k: float(np.mean([per_query[qid][k] for qid in per_query])) for k in keys}
        return agg, per_query

evaluator = Evaluator(k_values=EVAL_K_VALUES)

# ── 6. RRF / JinaReranker / HybridRetriever classes ──────────────────
class ReciprocalRankFusion:
    def __init__(self, k=60.0, weights=None):
        self.k = k
        self.weights = weights

    def fuse(self, ranked_lists):
        weights = self.weights or [1.0] * len(ranked_lists)
        doc_scores = {}
        for w, ranked in zip(weights, ranked_lists):
            for rank, (doc_id, _) in enumerate(ranked, start=1):
                doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + w / (self.k + rank)
        return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)

class JinaReranker:
    def __init__(self, model_name=JINA_RERANKER_MODEL, device=None, max_length=JINA_MAX_LENGTH, batch_size=JINA_BATCH_DOCS):
        self.model_name = model_name
        self.device     = device or DEVICE
        self.max_length = max_length
        self.batch_size = min(batch_size, 64)
        self._model     = None
        self._tokenizer = None
        self._corpus    = {}
        self._ready     = False

    def _load_model(self):
        if self._model is None:
            try:
                self._tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
                self._model = AutoModelForSequenceClassification.from_pretrained(
                    self.model_name,
                    torch_dtype=torch.float16 if "cuda" in self.device else torch.float32,
                    trust_remote_code=True,
                )
                self._model.to(self.device)
                self._model.eval()
            except Exception as exc:
                raise RuntimeError(
                    f"Could not load {self.model_name}."
                ) from exc

    def build_index(self, doc_ids, corpus_raw, chunk_texts=None, chunk_to_doc=None):
        if chunk_texts is not None and chunk_to_doc is not None:
            doc_chunks = defaultdict(list)
            for ct, cd in zip(chunk_texts, chunk_to_doc):
                doc_chunks[cd].append(ct)
            for doc_id in doc_ids:
                self._corpus[doc_id] = ' '.join(doc_chunks.get(doc_id, [""]))
        else:
            for doc_id in doc_ids:
                doc = corpus_raw.get(doc_id, {})
                self._corpus[doc_id] = (doc.get('title','') + ' ' + doc.get('text','')).strip()
        self._load_model()
        self._ready = True
        logger.info(f"Jina reranker ready: {self.model_name} on {self.device}")

    def _normalize_scores(self, raw, n_docs):
        if isinstance(raw, torch.Tensor):
            raw = raw.detach().cpu().flatten().tolist()
        if isinstance(raw, list) and raw and isinstance(raw[0], dict):
            scores = [0.0] * n_docs
            for item in raw:
                idx = int(item.get("index", len(scores)))
                if 0 <= idx < n_docs:
                    scores[idx] = float(item.get("relevance_score", item.get("score", 0.0)))
            return scores
        if isinstance(raw, list):
            return [float(x) for x in raw[:n_docs]]
        return [float(raw)] * n_docs

    def _score_batch(self, query, documents):
        self._load_model()
        if hasattr(self._model, "rerank"):
            raw = self._model.rerank(query=query, documents=documents, max_length=self.max_length)
            return self._normalize_scores(raw, len(documents))
        prompts = [
            f"<|im_start|>system\nYou are a search relevance expert.\n<|im_end|>\n"
            f"<|im_start|>user\nQuery: {query}\nDocument: {doc}\n<|im_end|>"
            for doc in documents
        ]
        enc = self._tokenizer(prompts, max_length=self.max_length, truncation=True, padding=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self._model(**enc)
            logits = out.logits.squeeze(-1)
        return logits.detach().float().cpu().tolist()

    def _score_single(self, query, document):
        return float(self._score_batch(query, [document])[0])

    def rerank(self, query, candidates, top_n=RERANKER_TOP_N):
        if not self._ready or not candidates:
            return candidates
        target = candidates[:top_n]
        remaining = candidates[top_n:]
        scored = []
        for start in range(0, len(target), self.batch_size):
            batch = target[start:start + self.batch_size]
            docs = [self._corpus.get(doc_id, "") for doc_id, _ in batch]
            scores = self._score_batch(query, docs)
            scored.extend((doc_id, float(score)) for (doc_id, _), score in zip(batch, scores))
        reranked = sorted(scored, key=lambda x: x[1], reverse=True)
        assert {d for d, _ in reranked} == {d for d, _ in target}
        return reranked + list(remaining)

class HybridRetriever:
    def __init__(self, sparse, dense, rrf, processor, reranker=None,
                 fetch_k=FETCH_K, bm25_weight=BM25_WEIGHT, dense_weight=DENSE_WEIGHT):
        self.sparse = sparse; self.dense = dense; self.rrf = rrf
        self.processor = processor; self.reranker = reranker
        self.fetch_k = fetch_k
        self.rrf.weights = [bm25_weight, dense_weight]
        self._tok_cache = {}

    def retrieve(self, query, top_k=100):
        qtoks = self._tok_cache.get(query)
        if qtoks is None:
            qtoks = self.processor.tokenize(query)
            self._tok_cache[query] = qtoks
        sparse_res = self.sparse.search(qtoks, self.fetch_k)
        dense_res  = self.dense.search(query, self.fetch_k)
        fused = self.rrf.fuse([sparse_res, dense_res])
        if self.reranker is not None:
            fused = self.reranker.rerank(query, fused, top_n=RERANKER_TOP_N)
        return fused[:top_k]

    def evaluate(self, queries, qrels, evaluator, top_k=100):
        all_results = {}
        desc = 'Hybrid + Rerank' if self.reranker else 'Hybrid'
        for qid in tqdm(queries.keys(), desc=desc):
            all_results[qid] = self.retrieve(queries[qid], top_k=top_k)
        return evaluator.evaluate_run(all_results, qrels)

# ── 7. Optimized grid-search parameters ──────────────────────────────────────
BEST_RRF_K   = 3
BEST_BM25_W  = 0.8
BEST_DENSE_W = 0.2
BEST_POOL    = 100

print("\n✓ Standalone setup complete — ready for LoRA fine-tuning.")

---
## Section 19 — LoRA Fine-Tuning on EnronQA

Construct pointwise training pairs with hard negatives and random negatives. Fine-tune using `peft` and custom manual training loop.

In [ ]:
# ── Step 1: Build training triples for LoRA ───────────────────────────────────
print("Building training triples with BM25 hard negatives and random negatives...")

train_examples_lora = []
n_positives = 0
n_hard_neg = 0
n_rand_neg = 0

for pair in tqdm(train_pairs_raw[:5000], desc="Mining hard negatives"):  # Cap at 5K queries for speed
    q_text = pair["query"]
    gold_id = pair["doc_id"]

    # Positive pair
    gold_text = processor.combine_fields_dense(corpus_raw.get(gold_id, {}))[:3000]
    train_examples_lora.append((q_text, gold_text, 1.0))
    n_positives += 1

    # Hard negatives: BM25 top-10 non-gold documents
    qtoks = processor.tokenize(q_text)
    bm25_results = sparse_retriever.search(qtoks, top_k=10)
    hard_neg_count = 0
    for neg_id, _ in bm25_results:
        if neg_id == gold_id:
            continue
        neg_text = processor.combine_fields_dense(corpus_raw.get(neg_id, {}))[:3000]
        train_examples_lora.append((q_text, neg_text, 0.0))
        n_hard_neg += 1
        hard_neg_count += 1
        if hard_neg_count >= 3:
            break

    # 1 random negative
    rand_id = random.choice(doc_ids)
    while rand_id == gold_id:
        rand_id = random.choice(doc_ids)
    rand_text = processor.combine_fields_dense(corpus_raw.get(rand_id, {}))[:3000]
    train_examples_lora.append((q_text, rand_text, 0.0))
    n_rand_neg += 1

print(f"\nTraining triples: {len(train_examples_lora):,}")
print(f"  Positives              : {n_positives:,}")
print(f"  BM25 hard negatives    : {n_hard_neg:,}")
print(f"  Random negatives       : {n_rand_neg:,}")
print(f"  Pos:Neg ratio          : 1:{(n_hard_neg+n_rand_neg)/max(n_positives,1):.1f}")

In [ ]:
# ── Step 2: Dataset, Evaluator, and Fine-tuning Helper Functions ─────────────
def tokenize_pair(query, document, tokenizer, max_length=512):
    """Tokenize a pointwise relevance prompt for jina-reranker-v3 LoRA training."""
    prompt = (
        f"<|im_start|>system\nYou are a search relevance expert.\n<|im_end|>\n"
        f"<|im_start|>user\nQuery: {query}\nDocument: {document}\n<|im_end|>"
    )
    return tokenizer(prompt, max_length=max_length, truncation=True, padding="max_length", return_tensors="pt")


class EnronRerankDataset(torch.utils.data.Dataset):
    """Dataset of (query, document, label) triples for LoRA."""
    def __init__(self, examples, tokenizer, max_length=512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        query, doc, label = self.examples[idx]
        enc = tokenize_pair(query, doc, self.tokenizer, self.max_length)
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float),
        }


def evaluate_lora_mrr(model, tokenizer, val_samples, device, max_length=FT_MAX_LENGTH, at_k=10):
    """Validation MRR@10 evaluator during training."""
    model.eval()
    rr = []
    with torch.no_grad():
        for sample in val_samples:
            docs = sample.get("positive", []) + sample.get("negative", [])
            labels = [1] * len(sample.get("positive", [])) + [0] * len(sample.get("negative", []))
            if not docs or not any(labels):
                continue
            scores = []
            for doc in docs[:at_k * 4]:
                enc = tokenize_pair(sample["query"], doc, tokenizer, max_length)
                enc = {k: v.to(device) for k, v in enc.items()}
                scores.append(float(model(**enc).logits.squeeze().detach().cpu()))
            ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
            rank = next((i + 1 for i, idx in enumerate(ranked[:at_k]) if labels[idx] == 1), None)
            rr.append(1.0 / rank if rank else 0.0)
    model.train()
    return float(np.mean(rr)) if rr else 0.0


def fine_tune_jina_lora(base_model_name, train_dataset, val_samples, lora_config, output_path, device,
                        num_epochs=FT_NUM_EPOCHS, batch_size=FT_BATCH_SIZE, lr=FT_LEARNING_RATE,
                        grad_accum_steps=FT_GRAD_ACCUM, warmup_ratio=FT_WARMUP_RATIO):
    """Fine-tune jina-reranker-v3 using LoRA."""
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=1,
        torch_dtype=torch.float16 if "cuda" in device else torch.float32,
        trust_remote_code=True,
    )
    peft_model = get_peft_model(base_model, lora_config)
    peft_model.print_trainable_parameters()
    peft_model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    optimizer = AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    total_steps = max(1, len(train_loader) * num_epochs // grad_accum_steps)
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    history = {"step": [], "loss": [], "val_mrr@10": []}
    best_mrr = -1.0
    global_step = 0
    os.makedirs(output_path, exist_ok=True)

    try:
        for epoch in range(num_epochs):
            peft_model.train()
            running_loss = 0.0
            optimizer.zero_grad(set_to_none=True)
            for step, batch in enumerate(tqdm(train_loader, desc=f"LoRA epoch {epoch+1}/{num_epochs}"), start=1):
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop("labels")
                outputs = peft_model(**batch)
                logits = outputs.logits.squeeze(-1).float()
                loss = loss_fn(logits, labels) / grad_accum_steps
                loss.backward()
                running_loss += float(loss.detach().cpu()) * grad_accum_steps

                if step % grad_accum_steps == 0 or step == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(peft_model.parameters(), max_norm=1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    global_step += 1

                    if global_step % 20 == 0:
                        avg_loss = running_loss / max(step, 1)
                        print(f"step={global_step} loss={avg_loss:.4f}")
                    if global_step % 200 == 0 or global_step == total_steps:
                        val_mrr = evaluate_lora_mrr(peft_model, tokenizer, val_samples, device)
                        avg_loss = running_loss / max(step, 1)
                        history["step"].append(global_step)
                        history["loss"].append(avg_loss)
                        history["val_mrr@10"].append(val_mrr)
                        print(f"validation MRR@10={val_mrr:.4f}")
                        if val_mrr > best_mrr:
                            best_mrr = val_mrr
                            peft_model.save_pretrained(output_path)
                            tokenizer.save_pretrained(output_path)
        peft_model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
    except KeyboardInterrupt:
        print("Interrupted. Saving current checkpoint...")
        peft_model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        raise

    with open(os.path.join(output_path, "training_metadata.pkl"), "wb") as f:
        pickle.dump({"history": history, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "examples": len(train_dataset)}, f)
    return output_path, history

print("Helper functions and dataset class defined.")

In [ ]:
# ── Step 3: Run Fine-tuning ──────────────────────────────────────────────────
# Prepare validation dataset from first 50 test queries
val_qids = list(queries_raw.keys())[:50]
val_samples_lora = []
for qid in val_qids:
    qtext = queries_raw[qid]
    gold_ids = set(qrels[qid].keys())
    qtoks = processor.tokenize(qtext)
    candidates = sparse_retriever.search(qtoks, top_k=20)
    pos_docs, neg_docs = [], []
    for did, _ in candidates:
        doc_text = processor.combine_fields_dense(corpus_raw.get(did, {}))[:2000]
        if did in gold_ids:
            pos_docs.append(doc_text)
        else:
            neg_docs.append(doc_text)
    if not pos_docs:
        for gid in gold_ids:
            pos_docs.append(processor.combine_fields_dense(corpus_raw.get(gid, {}))[:2000])
    val_samples_lora.append({"query": qtext, "positive": pos_docs, "negative": neg_docs})

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
)

tokenizer_for_ft = AutoTokenizer.from_pretrained(JINA_RERANKER_MODEL, trust_remote_code=True)
if tokenizer_for_ft.pad_token is None:
    tokenizer_for_ft.pad_token = tokenizer_for_ft.eos_token

# ── Launch fine-tuning (set to True to run) ──────────────────────────────────
RUN_LORA_TRAINING = False

if RUN_LORA_TRAINING:
    lora_adapter_path, lora_history = fine_tune_jina_lora(
        base_model_name=JINA_RERANKER_MODEL,
        train_dataset=EnronRerankDataset(train_examples_lora, tokenizer_for_ft, FT_MAX_LENGTH),
        val_samples=val_samples_lora,
        lora_config=lora_config,
        output_path=FT_OUTPUT_PATH,
        device=DEVICE,
        num_epochs=FT_NUM_EPOCHS,
        batch_size=FT_BATCH_SIZE,
        lr=FT_LEARNING_RATE,
        grad_accum_steps=FT_GRAD_ACCUM,
    )
    print(f"\n✓ LoRA adapters saved to {lora_adapter_path}")
    if lora_history.get("step"):
        fig, ax1 = plt.subplots(figsize=(8, 4))
        ax1.plot(lora_history["step"], lora_history["loss"], label="loss")
        ax1.set_ylabel("Loss")
        ax2 = ax1.twinx()
        ax2.plot(lora_history["step"], lora_history["val_mrr@10"], color="orange", label="MRR@10")
        ax2.set_ylabel("Validation MRR@10")
        plt.title("LoRA Training Curve")
        plt.savefig("lora_training_curve.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("RUN_LORA_TRAINING = False. Skipping training execution. Set to True to fine-tune.")

---
## Section 20 — Evaluate Fine-Tuned Jina Reranker v3 (LoRA)

In [ ]:
def load_finetuned_jina_reranker(base_model_name, lora_adapter_path, device):
    """Load a PEFT LoRA adapter, merge it into jina-reranker-v3, and return model/tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=1,
        torch_dtype=torch.float16 if "cuda" in device else torch.float32,
        trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_model, lora_adapter_path)
    merged_model = peft_model.merge_and_unload()
    merged_model.to(device)
    merged_model.eval()
    return merged_model, tokenizer

if os.path.isdir(FT_OUTPUT_PATH) and len(os.listdir(FT_OUTPUT_PATH)) > 0:
    ft_jina_model, ft_tokenizer = load_finetuned_jina_reranker(JINA_RERANKER_MODEL, FT_OUTPUT_PATH, DEVICE)
    ft_jina_reranker = JinaReranker(device=DEVICE)
    ft_jina_reranker._model = ft_jina_model
    ft_jina_reranker._tokenizer = ft_tokenizer
    ft_jina_reranker.build_index(doc_ids=doc_ids, corpus_raw=corpus_raw,
                                  chunk_texts=chunk_texts, chunk_to_doc=chunk_to_doc)

    s_high = ft_jina_reranker._score_single(
        "What contracts did Enron sign with California utilities?",
        "Enron Corporation entered into long-term energy contracts with Pacific Gas and Electric."
    )
    s_low = ft_jina_reranker._score_single(
        "What contracts did Enron sign with California utilities?",
        "The weather forecast for tomorrow shows sunny skies and mild temperatures."
    )
    print(f"Fine-tuned reranker sanity: relevant={s_high:.4f}, irrelevant={s_low:.4f}")
    assert s_high > s_low, "Fine-tuned Jina sanity check failed."

    rrf_final = ReciprocalRankFusion(k=BEST_RRF_K, weights=[BEST_BM25_W, BEST_DENSE_W])
    hybrid_ft = HybridRetriever(
        sparse=sparse_retriever, dense=dense_retriever, rrf=rrf_final,
        processor=processor, reranker=ft_jina_reranker,
        fetch_k=BEST_POOL, bm25_weight=BEST_BM25_W, dense_weight=BEST_DENSE_W,
    )
    print("\nEvaluating Hybrid + Fine-Tuned Jina Reranker v3 (LoRA)...")
    ft_agg, ft_per_q = hybrid_ft.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)
    print("\n=== Hybrid + Fine-Tuned Jina Reranker v3 (LoRA) ===")
    for m, v in sorted(ft_agg.items()):
        print(f"  {m:<20}: {v:.4f}")
else:
    print(f"No LoRA adapter found at {FT_OUTPUT_PATH}. Using off-the-shelf scores as placeholder for summary.")
    ft_agg, ft_per_q = offshelf_agg, offshelf_per_q
    hybrid_ft = hybrid_jina_offshelf

In [ ]:
# ── Hybrid without reranker (for fair comparison) ────────────────────────────
rrf_norank = ReciprocalRankFusion(k=BEST_RRF_K, weights=[BEST_BM25_W, BEST_DENSE_W])
hybrid_norank = HybridRetriever(
    sparse=sparse_retriever, dense=dense_retriever, rrf=rrf_norank,
    processor=processor, reranker=None,
    fetch_k=BEST_POOL, bm25_weight=BEST_BM25_W, dense_weight=BEST_DENSE_W,
)

print("Evaluating Hybrid (no reranker, optimized params)...")
norank_agg, norank_per_q = hybrid_norank.evaluate(queries_raw, qrels, evaluator, top_k=TOP_K)

---
## Section 21 — Final Comparison Summary

In [ ]:
# ── Re-evaluate BM25 with best params ────────────────────────────────────────
bm25_opt_agg, _ = sparse_retriever.evaluate(queries_raw, qrels, processor, evaluator, top_k=TOP_K)

# ── Final summary table ──────────────────────────────────────────────────────
SUMMARY_METRICS = ['mrr', 'ndcg@10', 'ndcg@100', 'recall@10', 'recall@100', 'precision@10']

summary_rows = [
    {'System': 'BM25 (optimized)',                            **{m: round(bm25_opt_agg.get(m,0),4) for m in SUMMARY_METRICS}},
    {'System': 'Dense BGE',                                   **{m: round(dense_agg.get(m,0),4) for m in SUMMARY_METRICS}},
    {'System': 'Hybrid RRF (no rerank)',                      **{m: round(norank_agg.get(m,0),4) for m in SUMMARY_METRICS}},
    {'System': 'Hybrid + Jina Reranker v3 (off-shelf)',       **{m: round(offshelf_agg.get(m,0),4) for m in SUMMARY_METRICS}},
    {'System': 'Hybrid + Jina Reranker v3 (fine-tuned LoRA)', **{m: round(ft_agg.get(m,0),4) for m in SUMMARY_METRICS}},
]

summary_df = pd.DataFrame(summary_rows).set_index('System')
print("\n" + "="*100)
print("FINAL SUMMARY — EnronQA Hybrid IR System")
print("="*100)
print(summary_df.to_string())

# Highlight best NDCG@10
best_system = summary_df['ndcg@10'].idxmax()
best_ndcg = summary_df['ndcg@10'].max()
print(f"\n>>> Best system: {best_system} (NDCG@10 = {best_ndcg:.4f})")
if best_ndcg >= 0.75:
    print("✓ TARGET ACHIEVED: NDCG@10 ≥ 0.75")
else:
    print(f"✗ Target not yet met: {best_ndcg:.4f} < 0.75 (gap: {0.75-best_ndcg:.4f})")

In [ ]:
# ── Final comparison visualization ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors_final = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
system_labels = ['BM25\n(optimized)', 'Dense\nBGE', 'Hybrid\n(no rerank)',
                 'Hybrid+Jina\n(off-shelf)', 'Hybrid+Jina\n(fine-tuned)']

for ax, metric in zip(axes, SUMMARY_METRICS):
    vals = [summary_rows[i].get(metric, 0) for i in range(5)]
    bars = ax.bar(range(5), vals, color=colors_final, edgecolor='white')
    ax.bar_label(bars, fmt='%.4f', padding=2, fontsize=8)
    ax.set_xticks(range(5))
    ax.set_xticklabels(system_labels, fontsize=7)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.grid(axis='y', alpha=0.4)
    # Add target line for NDCG@10
    if metric == 'ndcg@10':
        ax.axhline(y=0.75, color='red', linestyle='--', alpha=0.7, label='Target: 0.75')
        ax.legend(fontsize=8)

plt.suptitle('Final System Comparison — EnronQA IR', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 22 — Failure & Latency Analysis

In [ ]:
# ── Failure analysis ──────────────────────────────────────────────────────────
best_hybrid = hybrid_ft if ft_agg.get('ndcg@10',0) > norank_agg.get('ndcg@10',0) else hybrid_norank

bm25_res, dense_res, hybrid_res = {}, {}, {}
for qid, qtext in tqdm(queries_raw.items(), desc='Collecting results'):
    qtoks = processor.tokenize(qtext)
    bm25_res[qid] = sparse_retriever.search(qtoks, top_k=10)
    dense_res[qid] = dense_retriever.search(qtext, top_k=10)
    hybrid_res[qid] = best_hybrid.retrieve(qtext, top_k=10)

def hits_at_k(results, relevant, k=10):
    return int(bool({d for d,_ in results[:k]} & relevant))

failures = []
for qid in queries_raw:
    rel = set(qrels.get(qid, {}).keys())
    if not rel: continue
    failures.append({
        'qid': qid,
        'bm25': hits_at_k(bm25_res.get(qid,[]), rel),
        'dense': hits_at_k(dense_res.get(qid,[]), rel),
        'hybrid': hits_at_k(hybrid_res.get(qid,[]), rel),
    })

only_bm25_fail = [f for f in failures if f['bm25']==0 and f['dense']==1]
only_dense_fail = [f for f in failures if f['dense']==0 and f['bm25']==1]
hybrid_unique = [f for f in failures if f['hybrid']==1 and f['bm25']==0 and f['dense']==0]
all_fail = [f for f in failures if f['bm25']==0 and f['dense']==0 and f['hybrid']==0]

print(f"Total queries         : {len(failures)}")
print(f"BM25 fails, Dense hits: {len(only_bm25_fail)}")
print(f"Dense fails, BM25 hits: {len(only_dense_fail)}")
print(f"Hybrid unique wins    : {len(hybrid_unique)}")
print(f"All systems fail      : {len(all_fail)}")

# Show example failures
print("\n=== Example Failures (All Systems Miss) ===")
for f in all_fail[:5]:
    qid = f['qid']
    print(f"\n  Query: {queries_raw[qid][:100]}")
    rel = list(qrels[qid].keys())
    print(f"  Gold: {rel[0]}")
    gold_title = corpus_raw[rel[0]].get('title','')
    print(f"  Gold subject: {gold_title[:80]}")

In [ ]:
# ── Latency analysis ──────────────────────────────────────────────────────────
print("Measuring latencies...")
bm25_lat = sparse_retriever.measure_latency(queries_raw, processor)
dense_lat = dense_retriever.measure_latency(queries_raw)

def measure_hybrid_latency(hybrid, queries, n_queries=20):
    sample = list(queries.values())[:n_queries]
    latencies = []
    for qtext in sample:
        t0 = time.perf_counter()
        hybrid.retrieve(qtext, top_k=TOP_K)
        latencies.append((time.perf_counter()-t0)*1000)
    return {'mean_ms': round(np.mean(latencies),2), 'median_ms': round(np.median(latencies),2), 'p95_ms': round(np.percentile(latencies,95),2)}

latency_df = pd.DataFrame([
    {'System': 'BM25S (optimized)', **bm25_lat},
    {'System': 'Dense BGE', **dense_lat},
    {'System': 'JinaReranker (off-shelf, hybrid incl. retrieval)', **measure_hybrid_latency(hybrid_jina_offshelf, queries_raw)},
    {'System': 'JinaReranker (LoRA, hybrid incl. retrieval)', **measure_hybrid_latency(hybrid_ft, queries_raw)},
])
print("\n=== Latency Analysis (ms per query) ===")
print(latency_df.to_string(index=False))

In [ ]:
print("\n" + "="*70)
print("OPTIMIZATION CONFIGURATION SUMMARY")
print("="*70)
print(f"  Dense Model      : {SBERT_MODEL}")
print(f"  Embedding Dim    : 768")
print(f"  Query Prefix     : BGE instruction prefix enabled")
print(f"  BM25 k1          : {BEST_BM25_K1}")
print(f"  BM25 b           : {BEST_BM25_B}")
print(f"  RRF k            : {BEST_RRF_K}")
print(f"  BM25 weight      : {BEST_BM25_W}")
print(f"  Dense weight     : {BEST_DENSE_W}")
print(f"  Candidate pool   : {BEST_POOL}")
print(f"  Reranker         : {JINA_RERANKER_MODEL} (LoRA target: {LORA_TARGET})")
print(f"  FAISS index      : HNSW efSearch=128, METRIC_INNER_PRODUCT")
print(f"  Preprocessing    : Email-specific (thread-aware chunking, field boosting)")
print("="*70)

---
## Section 23 — Discussion

### Technical Advancements of Jina Reranker v3 (LoRA)

1. **Last But Not Late (LBNL) Causal Attention**:
   - In contrast to late interaction models (like ColBERT) that encode documents independently before vector cross-matching, Jina Reranker v3 performs full causal attention over the query and candidates in a shared context window.
   - This allows rich inter-document context and query-document interactions before extracting the final embedding.
   - Points out the advantage over traditional bi-encoders, especially in multi-hop or argumentative tasks.

2. **Parameter-Efficient Adapter Adaptation (PEFT / LoRA)**:
   - Rather than tuning the entire 0.6B backbone, we attach low-rank adapters to the self-attention projections (`q_proj`, `v_proj`, etc.) and intermediate feed-forward layers.
   - Pointwise classification objectives are fine-tuned using standard `BCEWithLogitsLoss`.

3. **Comparison with Cross-Encoder (monoMiniLM)**:
   - Jina Reranker v3 achieves higher baseline representation capacity (0.6B parameters vs. 22M for MiniLM).
   - Pointwise fine-tuning using mined hard negatives adapts Jina's pre-trained attention paths to corporate email structure.
   - Concatenating multi-message threads up to a sequence length of 1024 allows the model to view the complete history of long email exchanges.